In [21]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import time



In [5]:
url = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"

df = pd.read_csv(url, sep="\s+", header=None, names=["Year", "Month", "ONI"])
df

<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
/var/folders/xr/wnljw_mx1k5cgd14sjw7ssz80000gn/T/ipykernel_17379/1711187612.py:3: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(url, sep="\s+", header=None, names=["Year", "Month", "ONI"])


,Year,Month,ONI
SEAS,YR,TOTAL,ANOM
DJF,1950,24.72,-1.53
JFM,1950,25.17,-1.34
FMA,1950,25.75,-1.16
MAM,1950,26.12,-1.18
...,...,...,...
SON,2025,26.16,-0.51
OND,2025,26.05,-0.55
NDJ,2025,25.97,-0.54
DJF,2026,26.13,-0.37


In [8]:
df.isna().sum()

Year     0
Month    0
ONI      0
dtype: int64

In [3]:
"""
Scraper for NOAA ONI data using BeautifulSoup.
"""

from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup


ONI_URL = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"


SEASON_TO_MONTH = {
    "DJF": 1,
    "JFM": 2,
    "FMA": 3,
    "MAM": 4,
    "AMJ": 5,
    "MJJ": 6,
    "JJA": 7,
    "JAS": 8,
    "ASO": 9,
    "SON": 10,
    "OND": 11,
    "NDJ": 12,
}


def scrape_oni_with_bs4(url=ONI_URL):
    """
    Scrape NOAA ONI data.

    Returns
    -------
    pandas.DataFrame
        DataFrame with date, season, year, total, and oni columns.
    """
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text()

    rows = []

    for line in text.splitlines():
        parts = line.strip().split()

        if len(parts) != 4:
            continue

        season, year, total, anom = parts

        if season not in SEASON_TO_MONTH:
            continue

        rows.append({
            "season": season,
            "year": int(year),
            "month": SEASON_TO_MONTH[season],
            "total": float(total),
            "oni": float(anom),
        })

    df = pd.DataFrame(rows)

    df["date"] = pd.to_datetime(
        df["year"].astype(str) + "-" + df["month"].astype(str) + "-01",
        errors="coerce",
    )

    df = df[["date", "season", "year", "month", "total", "oni"]]
    df = df.sort_values("date").reset_index(drop=True)

    return df


def save_oni_data(output_path="data/raw/oni_data.csv"):
    """
    Scrape and save ONI data.
    """
    df = scrape_oni_with_bs4()

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(output_path, index=False)

    return df


if __name__ == "__main__":
    oni_df = save_oni_data()
    print(oni_df.head())

        date season  year  month  total   oni
0 1950-01-01    DJF  1950      1  24.72 -1.53
1 1950-02-01    JFM  1950      2  25.17 -1.34
2 1950-03-01    FMA  1950      3  25.75 -1.16
3 1950-04-01    MAM  1950      4  26.12 -1.18
4 1950-05-01    AMJ  1950      5  26.32 -1.07
